# Adaptive mapping workflow test 2

This test is using RESTORE+ dataset using Jawa-Bali as the AOI.

In [ ]:
#This code is used if the notebook is implemented in github codespace. Just remove the (#)
!python -m pip install .. --quiet

# Import GEE account setting from luma-stack

In [ ]:
# Use EE initialization from luma_ge
import ee 
import luma_ge

# Autheticate using service account (json file)

service_account_path = '../auth/ee-epstm2024.json'
success = luma_ge.initialize_with_service_account(service_account_path)

if success:
    print("Earth Engine initialized with service account successfully!")
else:
    print("Service account initialization failed. Try to authenticate earth engine manually")

#Check authentication status
status = luma_ge.get_auth_status()
print(f"Initialized: {status['initialized']}")
print(f"Authenticated: {status['authenticated']}")
if status['project']:
    print(f"Project: {status['project']}")


# Upload AOI

AOI is imported from RESTORE+ asset of Jawa-Bali region

In [ ]:
region_name = 'JawaBali'
regions_fc   = ee.FeatureCollection(
        "users/hadicu06/IIASA/RESTORE/vector_datasets/classification_regions")
aoi = (regions_fc
                   .filter(ee.Filter.eq('region_name', region_name))
            )

# Collect satellite images

In [ ]:
year = 2018

annual_col = (ee.ImageCollection("LANDSAT/COMPOSITES/C02/T1_L2_ANNUAL")
                    .filterDate(f'{year}-01-01', f'{year}-12-31'))
annual_img = annual_col.first()
composite  = annual_img.select(
        ['blue', 'green', 'red', 'nir', 'swir1', 'swir2'],
        ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']
    ).clip(aoi)

ndvi = composite.normalizedDifference(['nir', 'red']).rename('ndvi')
ndwi = composite.normalizedDifference(['green', 'nir']).rename('ndwi')
ndbi = composite.normalizedDifference(['swir1', 'nir']).rename('ndbi')
savi = composite.expression(
        '1.5 * (NIR - RED) / (NIR + RED + 0.5)',
        {'NIR': composite.select('nir'), 'RED': composite.select('red')}
    ).rename('savi')
evi = composite.expression(
    '2.5 * (NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1)',
    {'NIR': composite.select('nir'), 'RED': composite.select('red'), 'BLUE': composite.select('blue')}
  ).rename('evi')

stacked_landsat = composite.addBands([ndvi, ndwi, ndbi, savi, evi])

# Step 1a: Identify elements and properties
This step will identify the elements and properties that were defined in Step 0.
This step will be skipped in the testing and the identification will be done manually

# Step 1b: Classification Ruleset

Define all the functions to turn class definition into rulesets. Two schemes are used as comparisons

In [ ]:
from luma_ge.modular_workflow import load_hierarchical_scheme

tree = load_hierarchical_scheme("../data/test_data/RESTORE_test/hierarchical_scheme_restore.csv")

# Step 2: Build modular reference data

This step uploads the reference data that contains UML information inside its attribute table

## Option 1: Turn RESTORE+ training data to become a modular reference data

This section performs the following functions:
1. Retain the RESTORE+ raw training dataset (`simplified_class_crowdsourced`), sampled inside an AOI, and merge them into one file
2. Clean up the raw dataset
3. Enrich the dataset by sampling information on the manually defined elements and properties (according to the scheme definition) from external sources

This part is optional, can also be preparred manually outside the workflow

In [ ]:
# RUN THIS ONLY ONCE TO LOAD AND MERGE THE TRAINING DATA FROM RESTORE+ (IT TAKES A LONG TIME TO LOAD)


# import geopandas as gpd
# import pandas as pd
# from shapely.geometry import shape

# # Load training data from RESTORE+

# asset_dict = {
#     "c1": {
#         "path": "users/hadicu06/IIASA/RESTORE/training_samples_features/simplified_class_crowdsourced/country/01_undisturbedForest",
#         "class_name": "undisturbedForest",
#         "class_id": 1
#     },
#     "c2": {
#         "path": "users/hadicu06/IIASA/RESTORE/training_samples_features/simplified_class_crowdsourced/country/02_loggedOverForest",
#         "class_name": "loggedOverForest",
#         "class_id": 2
#     },
#     "c3": {
#         "path": "users/hadicu06/IIASA/RESTORE/training_samples_features/simplified_class_crowdsourced/country/03_oilPalmMonoculture",
#         "class_name": "oilPalm",
#         "class_id": 3
#     },
#     "c4": {
#         "path": "users/hadicu06/IIASA/RESTORE/training_samples_features/simplified_class_crowdsourced/country/04_treeBasedNotOilPalm",
#         "class_name": "treeBased",
#         "class_id": 4
#     },
#     "c5": {
#         "path": "users/hadicu06/IIASA/RESTORE/training_samples_features/simplified_class_crowdsourced/country/05_cropland",
#         "class_name": "cropland",
#         "class_id": 5
#     },
#     "c6": {
#         "path": "users/hadicu06/IIASA/RESTORE/training_samples_features/simplified_class_crowdsourced/country/06_shrub",
#         "class_name": "shrub",
#         "class_id": 6
#     },
#     "c7": {
#         "path": "users/hadicu06/IIASA/RESTORE/training_samples_features/simplified_class_crowdsourced/country/07_grassAndSavanna",
#         "class_name": "grassSavanna",
#         "class_id": 7
#     },
#     "c8": {
#         "path": "users/hadicu06/IIASA/RESTORE/training_samples_features/simplified_class_crowdsourced/country/08_waterbody",
#         "class_name": "water",
#         "class_id": 8
#     },
#     "c9": {
#         "path": "users/hadicu06/IIASA/RESTORE/training_samples_features/simplified_class_crowdsourced/country/09_settlement",
#         "class_name": "settlement",
#         "class_id": 9
#     },
#     "c10": {
#         "path": "users/hadicu06/IIASA/RESTORE/training_samples_features/simplified_class_crowdsourced/country/10_clearedLand",
#         "class_name": "cleared",
#         "class_id": 10
#     }
# }

# def ee_to_gdf_with_source(asset_path, class_name, class_id, source_id, batch_size=5000):
#     fc = ee.FeatureCollection(asset_path)

#     # Filter if the training data has a binary column indicating valid samples (e.g., "class_binary_str" == 1)
#     # fc = fc.filter(ee.Filter.eq("class_binary_str", 1))

#     size = fc.size().getInfo()
#     print(f"Filtered features in {source_id}: {size}")

#     if size == 0:
#         print(f"⚠️ No valid features in {source_id} after filtering.")
#         return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs="EPSG:4326")

#     rows = []

#     for start in range(0, size, batch_size):
#         batch = fc.toList(batch_size, start)
#         batch_info = batch.getInfo()

#         for f in batch_info:
#             if "geometry" not in f or f["geometry"] is None:
#                 continue

#             geom = shape(f["geometry"])
#             props = f.get("properties", {})

#             props["class_name"] = class_name
#             props["class_id"]   = class_id
#             props["source_id"]  = source_id

#             rows.append({**props, "geometry": geom})

#     if len(rows) == 0:
#         print(f"⚠️ No valid geometries extracted for {source_id}.")
#         return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs="EPSG:4326")

#     gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")
#     return gdf

# gdf_list = []

# for source_id, meta in asset_dict.items():
#     print(f"Loading {source_id}...")
    
#     gdf_tmp = ee_to_gdf_with_source(
#         asset_path = meta["path"],
#         class_name = meta["class_name"],
#         class_id   = meta["class_id"],
#         source_id  = source_id
#     )
    
#     gdf_list.append(gdf_tmp)

# # Merge everything
# merged_gdf = pd.concat(gdf_list, ignore_index=True)

# print("Total samples:", len(merged_gdf))

# print(merged_gdf["class_name"].value_counts())
# print(merged_gdf["source_id"].value_counts())

# merged_gdf.to_file("../data/test_data/training_data_restore.gpkg", driver="GPKG")


In [ ]:
# After loading and merging the training data from RESTORE+ (run the above code only once), load it and do some filtering/sampling for trial run

# from luma_ge.modular_workflow import load_modular_training_data

# data = load_modular_training_data(
#     shp_path="../data/test_data/training_data_restore.gpkg",
#     aoi=aoi
# )

# gdf = data["gdf"]

# # Filter samples
# gdf_filtered = gdf[gdf["class_binary_str"] == "1"].copy()

# # Sample up to 100 per class for trial run
# gdf_sampled = (
#     gdf_filtered
#     .groupby("class_name", group_keys=False)
#     .apply(lambda x: x.sample(n=min(len(x), 100), random_state=42))
#     .reset_index(drop=True)
# )

# print("Filtered samples:", len(gdf_filtered))
# print("Sampled samples:", len(gdf_sampled))
# print(gdf_sampled["class_name"].value_counts())

In [ ]:
# Load element to primitive mapping and enrich the training data with external sources of elements and properties

# from luma_ge.modular_workflow import (
#     load_element_mapping,
#     enrich_training_data,
# )

# mapping = load_element_mapping("../data/test_data/element_to_primitives.csv")
# print(mapping)

# result  = enrich_training_data(
#     gdf               = gdf_sampled,
#     mapping           = mapping,
#     binary_threshold  = 0.1,    # 10% cover = present
#     overwrite_existing= False,  # keep hand-labeled values
# )

# print("Enriched:", result["enriched"])
# print("Skipped :", result["skipped"])
# print("Failed  :", result["failed"])

## Option 2: Use pre-built modular training data

In [ ]:
from luma_ge.modular_workflow import load_modular_training_data

data = load_modular_training_data(
    shp_path="../data/test_data/RESTORE_test/JawaBali_RS_2018_targeted_allclasses1.shp",
    aoi=aoi
)

training_gdf = data["gdf"]

training_fc = data["ee_fc"]

# Sample up to 100 per class for trial run
gdf_sampled = (
    training_gdf
    .groupby("class_name", group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), 100), random_state=42))
    .reset_index(drop=True)
)


print("Sampled samples:", len(gdf_sampled))
print(gdf_sampled["class_name"].value_counts())



# Step 3a: Apply classification ruleset to training dataset

In [ ]:
# from luma_ge.modular_workflow import TrainingDataLabeller

# labeller_scheme1 = TrainingDataLabeller(
#     rules_df=scheme1_rules,
#     scheme_name="my_scheme",
#     nodata_value=0
# )

# scheme1_fc = labeller_scheme1.label(training_fc)

# labeller_scheme2 = TrainingDataLabeller(
#     rules_df=scheme2_rules,
#     scheme_name="my_scheme",
#     nodata_value=0
# )

# scheme2_fc = labeller_scheme2.label(training_fc)

In [ ]:
# Visualize the training data

# import geemap
# import ee

# m = geemap.Map()
# m.centerObject(aoi, 12)

# # Color palette (index = class_id)
# palette = [
#     "888780",  # 0 nodata
#     "1D9E75",  # 1
#     "378ADD",  # 2
#     "D85A30",  # 3
#     "BA7517",  # 4
#     "7F77DD",  # 5
#     "639922",  # 6
# ]

# # --- Scheme 1 styling ---
# scheme1_styled = scheme1_fc.map(
#     lambda f: f.set({
#         "style": {
#             "color": "black",
#             "pointSize": 5,
#             "fillColor": ee.List(palette).get(
#                 ee.Number(f.get("class_id")).int()
#             )
#         }
#     })
# )

# # --- Scheme 2 styling ---
# scheme2_styled = scheme2_fc.map(
#     lambda f: f.set({
#         "style": {
#             "color": "black",
#             "pointSize": 5,
#             "fillColor": ee.List(palette).get(
#                 ee.Number(f.get("class_id")).int()
#             )
#         }
#     })
# )

# # --- Add layers ---
# m.addLayer(aoi, {}, "AOI")

# m.addLayer(
#     scheme1_styled.style(**{"styleProperty": "style"}),
#     {},
#     "Scheme 1 (class_id)"
# )

# m.addLayer(
#     scheme2_styled.style(**{"styleProperty": "style"}),
#     {},
#     "Scheme 2 (class_id)"
# )

# # --- Build legend from rules_df ---
# # Use scheme1_rules (or whichever rules correspond)
# legend_dict = {"0": "nodata"}

# for _, row in scheme1_rules.iterrows():
#     legend_dict[str(int(row["class_id"]))] = row["class_name"]

# # Match colors to class IDs
# legend_colors = palette[:len(legend_dict)]

# # Add legend
# m.add_legend(
#     title="Class ID",
#     legend_dict=legend_dict,
#     colors=legend_colors
# )

# m

# Step 3b: Generate primitive layers

## Probabilistic primitive layers

In [ ]:
from luma_ge.modular_workflow import PrimitiveLayerTrainer

# predictor image
image = stacked_landsat

# modular training data
roi = training_fc

# Generate primitive layers

trainer = PrimitiveLayerTrainer(stacked_landsat, training_fc)

primitive_layers_mc = trainer.train_all_mc()

# Check the generated primitive layers

print(primitive_layers_mc.keys())

### Visualize probabilistic primitives

In [ ]:
# Visualize probabilistic primitive layers

m = geemap.Map()

m.centerObject(aoi, 12)

m.addLayer(aoi, {}, "AOI")


m.addLayer(
    training_fc,
    {"color": "red"},
    "Training points"
)

m.addLayer(
    primitive_layers_mc["tree_pres"],
    {"min": 0, "max": 1, "palette": ["white", "green"]},
    "tree_pres"
)
m.addLayer(
    primitive_layers_mc["buil_pres"],
    {"min": 0, "max": 1, "palette": ["white", "orange"]},
    "buil_pres"
)
m.addLayer(
    primitive_layers_mc["water_pres"],
    {"min": 0, "max": 1, "palette": ["white", "blue"]},
    "water_pres"
)

m

In [ ]:
# 1. Stack dict → ee.Image
primitive_stack_mc = ee.Image.cat(list(primitive_layers_mc.values()))
band_names = primitive_stack_mc.bandNames().getInfo()
print("Bands in probabilistic stack:", band_names)

# 2. Centroid needs a maxError argument when geometry comes from a shapefile
test_point = aoi.geometry().centroid(maxError=1)

# 3. Sample one pixel to verify probability output
sample = primitive_stack_mc.sample(
    region=test_point,
    scale=30,
    numPixels=1
).first().toDictionary().getInfo()

# Sample a subset of pixels inside AOI (adjust numPixels as needed)
sample_fc = primitive_stack_mc.sample(
    region=aoi,
    scale=30,
    numPixels=5000,   # or more, but keep it reasonable
    geometries=False
)

# Bring sampled data to client as a list of dicts
samples = sample_fc.getInfo()["features"]

# Convert to per‑band arrays
vals = {b: [] for b in band_names}
for f in samples:
    d = f["properties"]
    for b in band_names:
        vals[b].append(d[b])

print("Sample pixel values:")
for band, val in sample.items():
    status = "OK" if 0 < val < 1 else "BINARY — not probability!"
    print(f"  {band}: {val:.4f}  {status}")


import matplotlib.pyplot as plt
import numpy as np
import ee

# Convert to per‑band arrays
vals = {b: [] for b in band_names}
for f in samples:
    d = f["properties"]
    for b in band_names:
        vals[b].append(d[b])

# Plot histograms of probability values for each primitive
fig, axes = plt.subplots(1, len(band_names), figsize=(5*len(band_names), 4))
if len(band_names) == 1:
    axes = [axes]

for ax, b in zip(axes, band_names):
    arr = np.array(vals[b])

    # ---- statistics ----
    mean = np.mean(arr)
    std = np.std(arr)
    minv = np.min(arr)
    maxv = np.max(arr)
    median = np.median(arr)

    # ---- histogram ----
    ax.hist(arr, bins=20, range=(0, 1), color="steelblue", edgecolor="black")

    ax.set_title(b)
    ax.set_xlabel("Probability")
    ax.set_ylabel("Pixel count")

    # ---- show stats on plot ----
    text = (
        f"mean={mean:.3f}\n"
        f"std={std:.3f}\n"
        f"median={median:.3f}\n"
        f"min={minv:.3f}\n"
        f"max={maxv:.3f}"
    )

    ax.text(
        0.98, 0.98,
        text,
        transform=ax.transAxes,
        verticalalignment="top",
        horizontalalignment="right",
        bbox=dict(facecolor="white", alpha=0.8)
    )

        # ---- also print stats to console ----
    print(f"Statistics for {b}:")
    print(f"  Mean: {mean:.3f}")
    print(f"  Std: {std:.3f}")
    print(f"  Median: {median:.3f}")
    print(f"  Min: {minv:.3f}")
    print(f"  Max: {maxv:.3f}\n")

plt.tight_layout()
plt.show()

## Optional: clean up metadata and get the elements and properties

In [ ]:
# Clean each primitive layer manually, removing all properties including system:index
primitive_layers_clean = {}
for k, v in primitive_layers.items():
    img = ee.Image(v).toFloat().rename(k).copyProperties(v, [])  # copy no properties
    primitive_layers_clean[k] = img

# Concatenate into a single image
primitive_image = ee.Image.cat(list(primitive_layers_clean.values()))

# List layer names for reference
# Original keys
layer_names = list(primitive_layers.keys())

# Manually remove "system:index" if it exists
if "system:index" in layer_names:
    layer_names.remove("system:index")

print(layer_names)

# Step 4: Classification

## Generate using Monte Carlo simulation and probabilistic layers

In [ ]:
from luma_ge.modular_workflow import HierarchicalRuleSetClassifier

results1 = HierarchicalRuleSetClassifier(
    primitive_image = primitive_stack_mc,
    scheme_df       = tree,
    aoi             = aoi,
)

results1.summary()   # inspect the tree before running

# Optional step: validation

In [ ]:
# Unpack MC outputs
map1_mode    = results1["mode_map"]
map1_entropy = results1["entropy_map"]
map1_probs   = results1["class_probs"]
 
map2_mode    = results2["mode_map"]
map2_entropy = results2["entropy_map"]
map2_probs   = results2["class_probs"]
 

In [ ]:
from luma_ge.modular_workflow import validate_monte_carlo

# Run validation for both schemes
validate_monte_carlo(results1, scheme1_rules, "scheme1",
                     entropy_threshold=0.5, scheme_label="Scheme 1")
validate_monte_carlo(results2, scheme2_rules, "scheme2",
                     entropy_threshold=0.5, scheme_label="Scheme 2")



In [ ]:
# Check what rules were actually generated for scheme 2
print(scheme2_rules[["class_id", "class_name", "rule", "priority"]].to_string())
print(scheme1_rules[["class_id", "class_name", "rule", "priority"]].to_string())
print(scheme2_rules[["class_id", "class_name", "rule", "priority"]].to_string())
print(scheme1_rules[["class_id", "class_name", "rule", "priority"]].to_string())

# Check what threshold values are in the rules
# A rule like "tree_pres > 0.7" will miss pixels where tree_pres=0.65 even though the pixel is clearly forested

# Sample the primitive values at nodata pixels to understand what's there
# band_arrays = classifier2_mc._get_band_arrays(scale=30)
# nodata_mask = (results2["mode_map"] == 0)

# print("Primitive values at nodata pixels:")
# for name, arr in band_arrays.items():
#     vals = arr[nodata_mask]
#     print(f"  {name}: mean={vals.mean():.3f}  "
#           f"min={vals.min():.3f}  max={vals.max():.3f}")

## Compare deterministic vs mc

In [ ]:
from luma_ge.modular_workflow import compare_det_vs_mc

# Step 2: compare deterministic vs MC side by side
compare_det_vs_mc(
    det_class_map = det_map1,
    mc_results    = results1,
    rules_df      = scheme1_rules,
    scheme_name   = "scheme1",
    scheme_label  = "Scheme 1",
)

compare_det_vs_mc(
    det_class_map = det_map2,
    mc_results    = results2,
    rules_df      = scheme2_rules,
    scheme_name   = "scheme2",
    scheme_label  = "Scheme 2",
)

In [ ]:
# If most values are between 0.3 and 0.7, the RF is uncertain everywhere
# and both methods are guessing rather than classifying
# for name, arr in band_arrays.items():
#     mid = ((arr > 0.3) & (arr < 0.7)).mean() * 100
#     print(f"{name}: {mid:.1f}% of pixels in uncertain zone [0.3, 0.7]")

## Generate using class-labelled training dataset (Step 3a)

In [ ]:
# Sample the image at the labelled points
sample = stacked_landsat.sampleRegions(
    collection=scheme1_fc,
    properties=["class_id"],  # use the labelled class_id
    scale=30,
    geometries=False
)

# Create a Random Forest classifier with 50 trees
classifier = ee.Classifier.smileRandomForest(50)

# Train the classifier on the sampled points
trained_classifier = classifier.train(
    features=sample,
    classProperty="class_id",
    inputProperties= stacked_landsat.bandNames()
)

# Classify the image
classified_step3a = stacked_landsat.classify(trained_classifier)

# Rename the output band to class_id for clarity
classified_step3a = classified_step3a.rename("class_id")

# Display a quick summary
print("Classification complete.")

In [ ]:
from luma_ge.modular_workflow import compare_rf_vs_mc

# Quick correction because step 3a produces different resolution than MC outputs (which also needs to be fixed later), so we need to resample the RF output to match the MC output shape
rf_array = geemap.ee_to_numpy(classified_step3a, region=roi, scale=30)
rf_array = rf_array.squeeze()  # remove band dimension

def resize_nearest(src, target_shape):
    src_h, src_w = src.shape
    tgt_h, tgt_w = target_shape

    row_idx = (np.linspace(0, src_h - 1, tgt_h)).astype(int)
    col_idx = (np.linspace(0, src_w - 1, tgt_w)).astype(int)

    return src[row_idx[:, None], col_idx]

rf_resampled = resize_nearest(rf_array, results1["mode_map"].shape)


In [ ]:

# Now we can compare the step 3a results (RF) to the MC results
compare_rf_vs_mc(
    rf_class_map=rf_resampled,
    mc_results=det_map1,
    rules_df=scheme1_rules,
    scheme_name="scheme1",
)